# Módulo 7 — Cointegração
## Seção 7.3 — Procedimento de Engle–Granger: regressão de longo prazo e teste residual

**Estado:** material didático para revisão científica. **Pré-requisitos:** séries $I(0)$ e $I(1)$; regressão por mínimos quadrados; testes ADF; Módulo 7, §§7.1–7.2. **Objetivo:** distinguir estimativa da relação de equilíbrio, teste de hipótese e identificação de dinâmica.

**Origem bibliográfica.** Morettin e Toloi, *Análise de Séries Temporais*, vol. 1 (2018), cap. 5, §5.3.2 (pp. 144–145), e Apêndice B (a partir da p. 411) são **pré-requisitos univariados** sobre integração e raízes unitárias. O procedimento de Engle–Granger propriamente dito **não foi localizado como tratamento específico nessa obra**. A teoria abaixo é complementação acadêmica baseada na abordagem de Engle e Granger (1987); seus detalhes devem ser cotejados com esse artigo original antes de uso em publicação.

### 1. Por que uma regressão usual não basta?

Se $X_t$ e $Y_t$ são $I(1)$, uma regressão em níveis pode apresentar $R^2$ elevado e testes $t$ convencionais enganosos, mesmo quando as séries são independentes (Seção 7.1). Em vez de avaliar apenas a inclinação estimada, Engle–Granger investiga se o **desvio em relação à regressão de longo prazo** é $I(0)$.

Considere

$$Y_t=\alpha+\beta X_t+u_t,\qquad X_t,Y_t\sim I(1).$$

A hipótese econômica de longo prazo é que $u_t$ não acumule permanentemente choques, isto é, $u_t\sim I(0)$. A direção da regressão define uma **normalização**: em amostras finitas, inverter dependente e regressora pode alterar resíduos e resultados do teste.

### 2. Etapa 1 — estimar a combinação de longo prazo

Em uma amostra de $n$ instantes, definimos

$$Q(a,b)=\sum_{t=1}^n(Y_t-a-bX_t)^2.$$

Derivando:

$$\frac{\partial Q}{\partial a}=-2\sum_t(Y_t-a-bX_t)=0,\quad
\frac{\partial Q}{\partial b}=-2\sum_tX_t(Y_t-a-bX_t)=0.$$

A primeira equação fornece $\widehat\alpha=\bar Y-\widehat\beta\bar X$. Substituindo na segunda, obtemos:

$$\boxed{\widehat\beta=\frac{\sum_t(X_t-\bar X)(Y_t-\bar Y)}{\sum_t(X_t-\bar X)^2}},\qquad\widehat\alpha=\bar Y-\widehat\beta\bar X.$$

As fórmulas exigem $\sum_t(X_t-\bar X)^2>0$. Calcule os **resíduos estimados** $\widehat u_t=Y_t-\widehat\alpha-\widehat\beta X_t$. Quando há cointegração sob condições adequadas, a inclinação de longo prazo pode ser superconsistente; isso **não** valida os erros-padrão usuais de uma regressão estática em séries integradas.

**Exemplo numérico (etapa 1).** Para $X=(1,2,3)$ e $Y=(4,5,8)$, temos $\bar X=2$, $\bar Y=17/3$, $\sum(X-\bar X)^2=2$, $\sum(X-\bar X)(Y-\bar Y)=4$, logo $\widehat\beta=2$ e $\widehat\alpha=5/3$. Os resíduos são $(1/3,-2/3,1/3)$. **Três observações não permitem decidir cointegração**; o exemplo ilustra apenas os cálculos de MQO.

### 3. Etapa 2 — testar raiz unitária no resíduo estimado

Uma regressão auxiliar representativa é:

$$\Delta\widehat u_t=\rho\widehat u_{t-1}+\sum_{j=1}^{k}\gamma_j\Delta\widehat u_{t-j}+e_t.$$

Aqui, $\rho=0$ corresponde à raiz unitária na parametrização de diferença, e $\rho<0$ é compatível com reversão estacionária. Adicionamos diferenças defasadas para lidar com autocorrelação serial; a escolha de $k$ influencia o resultado.

**Hipóteses estatísticas:** $H_0$: não cointegração; $H_1$: cointegração na relação determinística especificada. **Ponto fundamental:** como $\widehat u_t$ vem de uma regressão previamente estimada, a estatística não segue a distribuição do ADF padrão. Devem ser utilizados **valores críticos específicos do teste residual de cointegração**, que dependem do número de variáveis e das componentes determinísticas.

Na rotina `statsmodels.tsa.stattools.coint`, a opção `trend='c'` permite intercepto na relação de longo prazo; `trend='ct'`, tendência linear além do intercepto. O retorno é `(estatística, p_valor, valores_criticos)`. O parâmetro `autolag='aic'` escolhe defasagens segundo AIC dentro das restrições da função. Os $p$-valores são aproximações e o teste pode ser pouco potente em amostras pequenas.

### 4. Simulação controlada e execução em Python

As séries abaixo são geradas com sementes fixas. No primeiro par, $X_t$ é passeio aleatório e $Y_t=2+1{,}5X_t+u_t$, com $u_t=0{,}45u_{t-1}+\eta_t$; no segundo par, $Y_t$ é passeio aleatório independente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller

rng = np.random.default_rng(731)
n = 420
x = np.cumsum(rng.normal(size=n))
u = np.zeros(n)
eta = rng.normal(scale=0.8, size=n)
for t in range(1, n):
    u[t] = 0.45 * u[t-1] + eta[t]
y_coin = 2.0 + 1.5*x + u
y_ind = np.cumsum(rng.normal(size=n))

for nome, y in [('Cointegradas por construção', y_coin),
                ('Passeios independentes', y_ind)]:
    A = np.column_stack([np.ones(n), x])
    alfa, beta = np.linalg.lstsq(A, y, rcond=None)[0]
    res = y-(alfa+beta*x)
    eg_stat, eg_p, crit = coint(y, x, trend='c', autolag='aic')
    # Apenas demonstra o perigo de usar o p-valor ADF padrão no resíduo estimado.
    adf_stat, adf_p, *_ = adfuller(res, regression='n', autolag='AIC')
    print(f'{nome}: alfa={alfa:.3f}, beta={beta:.3f}')
    print(f'  EG: estat={eg_stat:.3f}, p={eg_p:.4f}, críticos={np.round(crit, 3)}')
    print(f'  ADF ingênuo do resíduo (não é o teste EG): p={adf_p:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
axes[0].plot(y_coin-2-1.5*x, label='Desvio cointegrado verdadeiro')
axes[1].plot(y_ind-x, label='Diferença entre dois passeios')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.25)
axes[1].set_xlabel('Tempo'); fig.tight_layout()
plt.savefig('figuras/07_03_desvios.png', dpi=145)
plt.show()

**Figura 7.3.1 —** O primeiro desvio tem distribuição estacionária por construção; a diferença entre os passeios independentes é $I(1)$. Um gráfico isolado não estabelece integração ou cointegração.

### 5. Fluxo de decisão e limitações

1. Inspecionar dados, frequência, quebras, tendência e sazonalidade.
2. Avaliar se as duas séries são plausivelmente $I(1)$, usando evidências teóricas, gráficos e testes de raiz unitária com seus limites.
3. Escolher **antes do teste** o sentido da regressão e sua especificação determinística.
4. Estimar MQO em níveis, obter resíduos e executar um teste residual com distribuição adequada.
5. Se houver evidência de cointegração, estimar uma dinâmica ECM e avaliar seus resíduos; se não houver, não inferir ausência definitiva de relação econômica.

**Cuidados:** regressores quase constantes, alta colinearidade, ordens de integração mistas, quebras estruturais e regimes instáveis prejudicam a interpretação. Para três ou mais séries, pode existir mais de um vetor cointegrante; isso motiva o procedimento de Johansen na Seção 7.5. **Cointegração não demonstra causalidade.**

### 6. Exercícios de autonomia

**A.** Derive ambas as equações normais a partir de $Q(a,b)$, sem consultar o resultado.

**B.** Repita o exemplo com $\beta=0{,}8$ e $\phi=0{,}85$. Faça pseudocódigo antes de alterar a simulação. Investigue sensibilidade a $n$ e às defasagens.

**C.** Explique por que o $p$-valor `adfuller(res)` não pode substituir `coint(y,x)` após a estimação de $a,b$.

**D.** Compare `trend='c'` e `trend='ct'`. Interprete por que as distribuições críticas mudam e por que a especificação não deve ser escolhida apenas pelo menor $p$-valor.

### Referências e alcance

**Livro principal (verificado como pré-requisito):** Morettin e Toloi (2018), cap. 5, §5.3.2, pp. 144–145; Apêndice B, pp. 411 ss. **Fundamento específico complementar:** Engle, R. F.; Granger, C. W. J. (1987), *Co-integration and error correction: Representation, estimation, and testing*, *Econometrica*, 55(2), 251–276. **Leitura complementar sugerida, sem cotejamento de páginas nesta produção:** Hamilton (1994), *Time Series Analysis*; Enders (2014), *Applied Econometric Time Series*. A dedução por MQO, os números simulados e os exercícios são desenvolvimentos didáticos desta seção.